In [1]:
# # ── Cell 0 ──────────────────────────────────────────────────────────────────
# # ╔══════════════════════════════════════════════════════════════════════════╗
# # ║  CELL 0 — Environment setup + GPU cleanup                               ║
# # ║  Run this FIRST every time. Kernel → Restart → Run All after this cell. ║
# # ╚══════════════════════════════════════════════════════════════════════════╝
# import subprocess, sys, gc

# def _pip(*args):
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(args))

# # ── 1. Pin every library to a known-good, tested combination ──────────────
# print("Installing pinned library stack ...")
# _pip("--upgrade",
#      "torch==2.3.1",
#      "torchvision==0.18.1",
#      "--index-url", "https://download.pytorch.org/whl/cu121")
# _pip(
#     "transformers==4.44.2",
#     "trl==0.10.1",
#     "peft==0.12.0",
#     "accelerate==0.34.2",
#     "bitsandbytes==0.43.3",
#     "datasets==2.21.0",
#     "huggingface_hub==0.24.6",
#     "sentencepiece",
#     "scipy",
#     "rich",
# )

# # ── 2. GPU cleanup ────────────────────────────────────────────────────────
# import torch
# for _var in ["model", "trainer", "ft_model", "base_model"]:
#     if _var in dir():
#         del globals()[_var]
# gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.reset_peak_memory_stats()

# # ── 3. Sanity check ───────────────────────────────────────────────────────
# import transformers, trl, peft, accelerate
# free  = torch.cuda.mem_get_info()[0] / 1e9
# total = torch.cuda.mem_get_info()[1] / 1e9

# print()
# print(f"torch        : {torch.__version__}")
# print(f"transformers : {transformers.__version__}")
# print(f"trl          : {trl.__version__}")
# print(f"peft         : {peft.__version__}")
# print(f"accelerate   : {accelerate.__version__}")
# print(f"CUDA         : {torch.version.cuda}")
# print(f"GPU          : {torch.cuda.get_device_name(0)}")
# print(f"VRAM free    : {free:.1f} / {total:.1f} GB")
# print()
# print("✅ Environment ready.")
# print("👉 NOW: Kernel → Restart Kernel, then Run All Cells")


# Fin-R1 (7B) — QLoRA Fine-Tuning for Multilingual Financial MCQ
## Running on Vast.ai A100 SXM4 (40 GB)

**Model**: `SUFE-AIFLM-Lab/Fin-R1` (Qwen2.5-7B-Instruct + finance SFT + GRPO)
**Method**: QLoRA — 4-bit NF4 + LoRA adapters (r=64, α=128)
**Data**: 1800 training examples across 6 multilingual sources (300 per source)
**Key improvements over baseline**:
- Choice order shuffled → eliminates positional bias (model was over-predicting A)
- Hindi questions translated to English → fixes 34.7% accuracy on hindi_finance
- r=64 LoRA → more capacity for 6-language adaptation
- BF16 training → cleaner gradients on A100
- 3 epochs with effective batch 32 → stable convergence

**Runtime**: ~25–35 min on A100 40GB
**Output**: Merged model pushed to HuggingFace Hub (loadable with AutoModelForCausalLM)

In [2]:
# ── Cell 2 ──────────────────────────────────────────────────────────────────

# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit before running
# ═══════════════════════════════════════════════════════════════════════════

TEST_MODE = False   # True = 4-step smoke test | False = full training run

# ── Model ──────────────────────────────────────────────────────────────────
MODEL_ID   = "SUFE-AIFLM-Lab/Fin-R1"  # TEST: same arch as Fin-R1, fits T4
             # Real run → "SUFE-AIFLM-Lab/Fin-R1"
OUTPUT_DIR = "/workspace/finr1_finetuned"   # Vast.ai local workspace
HUB_REPO   = "hassanshahzad2003/finr1-mcq-multilingual"  # HuggingFace Hub repo

# ── Data ───────────────────────────────────────────────────────────────────
RANDOM_SEED       = 42
N_TRAIN_PER_SRC   = 4   if TEST_MODE else 300  # 300 × 6 = 1800 total train
N_VAL_PER_SRC     = 2   if TEST_MODE else 50   # 50  × 6 = 300  total val
N_TEST_PER_SRC    = 2   if TEST_MODE else 80   # 80  × 6 = 480  total test

TRANSLATE_HINDI   = True   # Translate hindi_finance questions → English
                            # Fixes the 34.7% accuracy problem on Hindi
SHUFFLE_CHOICES   = True   # Randomly reorder answer choices per example
                            # Eliminates positional bias (model over-predicts A)

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_R               = 8  if TEST_MODE else 64   # r=64: more capacity for 6 languages
LORA_ALPHA           = 16 if TEST_MODE else 128  # always 2× rank
LORA_DROPOUT         = 0.05
LORA_TARGET_MODULES  = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# ── Training ───────────────────────────────────────────────────────────────
EPOCHS         = 1   if TEST_MODE else 3      # 3 epochs: format → multilingual → consolidate
BATCH_SIZE     = 2   if TEST_MODE else 2      # 7B BF16 full: batch=2 to avoid OOM
GRAD_ACCUM     = 2   if TEST_MODE else 16     # effective batch = 32 (2×16)
LEARNING_RATE  = 1e-4                         # lower than 2e-4 because r=64 has more capacity
LR_SCHEDULER   = "cosine"
WARMUP_RATIO   = 0.05
MAX_SEQ_LEN    = 256  if TEST_MODE else 512   # 512 saves ~40% VRAM vs 1024
SAVE_STEPS     = 5    if TEST_MODE else 50
LOGGING_STEPS  = 1    if TEST_MODE else 10
BF16           = True   # T4 has no BF16 support → use FP16
FP16           = False    # T4: FP16 works fine
                         # For A100 → flip: BF16=True, FP16=False
MAX_STEPS      = 4    if TEST_MODE else -1

# ── GitHub data source ─────────────────────────────────────────────────────
_GITHUB_REPO   = "hassan09070/clef_task"
_GITHUB_FOLDER = "task1_data"
_GITHUB_TOKEN  = ""   # read from env var GITHUB_TOKEN

print("Configuration loaded.")
if TEST_MODE:
    print("  ⚠  TEST_MODE = True — smoke test only")
print(f"  Model            : {MODEL_ID}")
print(f"  Train per source : {N_TRAIN_PER_SRC} × 6 = {N_TRAIN_PER_SRC * 6} total")
print(f"  Val per source   : {N_VAL_PER_SRC} × 6 = {N_VAL_PER_SRC * 6} total")
print(f"  Test per source  : {N_TEST_PER_SRC} × 6 = {N_TEST_PER_SRC * 6} total")
print(f"  Translate Hindi  : {TRANSLATE_HINDI}")
print(f"  Shuffle choices  : {SHUFFLE_CHOICES}")
print(f"  LoRA rank        : {LORA_R}  alpha={LORA_ALPHA}")
print(f"  Epochs           : {EPOCHS}  effective_batch={BATCH_SIZE * GRAD_ACCUM}")
print(f"  Learning rate    : {LEARNING_RATE}")
print(f"  Precision        : {'BF16' if BF16 else 'FP16' if FP16 else 'FP32'}")
print(f"  Output dir       : {OUTPUT_DIR}")


Configuration loaded.
  Model            : SUFE-AIFLM-Lab/Fin-R1
  Train per source : 300 × 6 = 1800 total
  Val per source   : 50 × 6 = 300 total
  Test per source  : 80 × 6 = 480 total
  Translate Hindi  : True
  Shuffle choices  : True
  LoRA rank        : 64  alpha=128
  Epochs           : 3  effective_batch=32
  Learning rate    : 0.0001
  Precision        : BF16
  Output dir       : /workspace/finr1_finetuned


In [3]:
# ── Cell 3 ──────────────────────────────────────────────────────────────────
import subprocess, sys

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

_pip("transformers>=4.40.0", "datasets", "peft", "trl>=0.8.0",
     "bitsandbytes>=0.46.1", "accelerate", "sentencepiece", "protobuf",
     "scikit-learn", "huggingface_hub", "deep-translator")

print("Libraries installed.")

Libraries installed.


In [4]:
# ── Cell 4 ──────────────────────────────────────────────────────────────────
import os
from huggingface_hub import login

# ── Output directory ───────────────────────────────────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# ── HuggingFace login ──────────────────────────────────────────────────────
hf_token = os.environ.get("HF_TOKEN", "YOUR_HF_TOKEN_HERE")
if hf_token:
    login(token=hf_token)
    print("HuggingFace login OK")
else:
    print("WARNING: HF_TOKEN env var not set — Hub push will fail")
    print("  Set it with: export HF_TOKEN=your_token_here")

# ── GitHub token ───────────────────────────────────────────────────────────
_GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "YOUR_GITHUB_TOKEN_HERE")
if _GITHUB_TOKEN:
    print("GitHub token loaded from GITHUB_TOKEN env var")
else:
    print("WARNING: GITHUB_TOKEN env var not set — data loading may fail")

# ── A100 optimisations ─────────────────────────────────────────────────────
import torch
torch.backends.cuda.matmul.allow_tf32 = True   # free speedup on Ampere GPUs
torch.backends.cudnn.allow_tf32       = True
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Output directory: /workspace/finr1_finetuned
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /root/.cache/huggingface/token
Login successful
HuggingFace login OK
GitHub token loaded from GITHUB_TOKEN env var
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.3 GB


In [5]:
# ── Cell 5 ──────────────────────────────────────────────────────────────────

import requests
import json
import base64
import os

# ── GitHub source config ────────────────────────────────────────────────────────────────────

def _get_github_token() -> str:
    # No Colab — read from environment variable only
    return os.environ.get("GITHUB_TOKEN") or _GITHUB_TOKEN


def _github_fetch(filename: str) -> list:
    """
    Fetch a JSON file from the private GitHub repo.
    Handles large files (>1 MB) via download_url streaming fallback.
    """
    token   = _get_github_token()
    headers = {
        "Accept":        "application/vnd.github+json",
        "Authorization": f"token {token}",
    }
    api_url = (
        f"https://api.github.com/repos/{_GITHUB_REPO}/contents/"
        f"{_GITHUB_FOLDER}/{filename}"
    )
    meta = requests.get(api_url, headers=headers, timeout=60)
    meta.raise_for_status()
    data = meta.json()

    content_b64 = data.get("content", "").replace("\n", "")
    if content_b64:
        return json.loads(base64.b64decode(content_b64).decode("utf-8"))

    # Large file path (>1 MB)
    download_url = data.get("download_url")
    if not download_url:
        raise ValueError(f"No content or download_url for {filename}")
    print(f"    [large file] streaming via download_url ...")
    chunks = []
    raw = requests.get(
        download_url,
        headers={"Authorization": f"token {token}"},
        timeout=300,
        stream=True,
    )
    raw.raise_for_status()
    total = 0
    for chunk in raw.iter_content(chunk_size=65536):
        chunks.append(chunk)
        total += len(chunk)
    print(f"    [large file] downloaded {total // 1024} KB")
    return json.loads(b"".join(chunks).decode("utf-8"))


# ── Dataset file mapping ───────────────────────────────────────────────────────────────
_DATASET_FILES = {
    "cfa_cpa":           "task1_Tomas08119993_finmmeval-cfa-cpa.json",
    "es_multifin":       "task1_TheFinAI_flare-es-multifin.json",
    "plutus":            "task1_TheFinAI_plutus-multifin.json",
    "arabic_accounting": "task1_SahmBenchmark_arabic-accounting-mcq.json",
    "arabic_business":   "task1_SahmBenchmark_arabic-business-mcq.json",
    "hindi_finance":     "task1_bharatgenai_BhashaBench-Finance-Hindi.json",
}


def load_task1_json(filename: str, source_name: str) -> list:
    records = _github_fetch(filename)
    rows    = []
    skipped = 0
    for rec in records:
        options = rec.get("options") or {}
        if not options:
            skipped += 1
            continue

        # Normalise all option keys to lowercase so gold comparison is
        # case-insensitive (some datasets use "A"/"B" keys, others "a"/"b").
        options_lc  = {k.lower(): v for k, v in options.items()}
        sorted_keys = sorted(options_lc.keys())          # always lowercase
        choices     = [options_lc[k] for k in sorted_keys]

        gold_raw   = rec.get("gold") or []
        valid_gold = [g.lower() for g in gold_raw if g.lower() in sorted_keys]
        if not valid_gold:
            skipped += 1
            continue

        rows.append({
            "question": str(rec.get("question") or ""),
            "choices":  choices,
            "keys":     sorted_keys,
            "gold":     valid_gold,
            "source":   source_name,
        })

    if skipped and not rows:
        # Help debug: show the first raw record so format issues are visible
        sample = records[0] if records else {}
        print(f"    ⚠  All {skipped} records skipped — gold key mismatch?")
        print(f"    Sample record keys : {list(sample.keys())}")
        if "options" in sample:
            print(f"    options keys       : {list(sample['options'].keys())}")
        if "gold" in sample:
            print(f"    gold value         : {sample['gold']!r}")
    elif skipped:
        print(f"    ({skipped} records skipped — no valid gold key)")

    return rows


# ── Load all datasets ────────────────────────────────────────────────────────────────────
print(f"Loading datasets from github.com/{_GITHUB_REPO} ...")
print()

datasets_raw = {}   # {source_name: list_of_rows}

for src_name, fname in _DATASET_FILES.items():
    print(f"  Loading {src_name} ...", flush=True)
    try:
        rows = load_task1_json(fname, src_name)
        if rows:
            datasets_raw[src_name] = rows
            print(f"  {src_name:<33} {len(rows):>5} rows  OK")
        else:
            print(f"  {src_name:<33}     0 rows  SKIPPED (see debug above)")
    except Exception as e:
        print(f"  {src_name:<33} FAILED — {e}")

print()
total = sum(len(r) for r in datasets_raw.values())
print(f"Total rows loaded: {total} across {len(datasets_raw)} sources")
if total == 0:
    print()
    print("  ⚠  No data loaded at all — likely GITHUB_TOKEN is missing or wrong.")
    print("  Check that the GITHUB_TOKEN env var is set on the Vast.ai instance.")
    raise RuntimeError("No dataset rows loaded — cannot continue. See warnings above.")
print("Note: train/val/test split happens in Cell 6 — all rows kept here.")


Loading datasets from github.com/hassan09070/clef_task ...

  Loading cfa_cpa ...
    [large file] streaming via download_url ...
    [large file] downloaded 1061 KB
  cfa_cpa                             768 rows  OK
  Loading es_multifin ...
  es_multifin                         220 rows  OK
  Loading plutus ...
  plutus                              246 rows  OK
  Loading arabic_accounting ...
  arabic_accounting                   405 rows  OK
  Loading arabic_business ...
  arabic_business                     374 rows  OK
  Loading hindi_finance ...
    [large file] streaming via download_url ...
    [large file] downloaded 5028 KB
  hindi_finance                      5982 rows  OK

Total rows loaded: 7995 across 6 sources
Note: train/val/test split happens in Cell 6 — all rows kept here.


In [6]:
# ── Cell 6 ──────────────────────────────────────────────────────────────────

# ═══════════════════════════════════════════════════════════════════════════
# Cell 5b — Cap / balance datasets
#
# TEST_MODE : keep only N_TRAIN + N_VAL + N_TEST rows per source (e.g. 8).
#             Translation then runs on at most that many Hindi rows.
# Full mode : cap all sources to the smallest source size so every
#             language contributes equally to training.
# ═══════════════════════════════════════════════════════════════════════════

import random as _bal_mod

_bal_rng = _bal_mod.Random(RANDOM_SEED)

print("Source sizes before capping:")
for src, rows in datasets_raw.items():
    print(f"  {src:<35} {len(rows):>5} rows")

if TEST_MODE:
    # Smoke test: grab only what the split cells will actually use.
    _cap = N_TRAIN_PER_SRC + N_VAL_PER_SRC + N_TEST_PER_SRC   # e.g. 4+2+2 = 8
    for src in list(datasets_raw.keys()):
        shuffled = datasets_raw[src][:]
        _bal_rng.shuffle(shuffled)
        datasets_raw[src] = shuffled[:_cap]
    print(f"\nTEST_MODE: capped each source to {_cap} rows "
          f"({N_TRAIN_PER_SRC} train + {N_VAL_PER_SRC} val + {N_TEST_PER_SRC} test)")
    print("Hindi translation (next cell) will only run on these few rows.")
else:
    # Full run: balance all sources to the smallest one.
    min_count = min(len(rows) for rows in datasets_raw.values())
    print(f"\nSmallest source: {min_count} rows — capping all sources to this.")
    for src in list(datasets_raw.keys()):
        shuffled = datasets_raw[src][:]
        _bal_rng.shuffle(shuffled)
        datasets_raw[src] = shuffled[:min_count]
    # Recalculate per-source split counts from the balanced size.
    N_TRAIN_PER_SRC = int(min_count * 0.75)
    N_VAL_PER_SRC   = int(min_count * 0.12)
    N_TEST_PER_SRC  = min_count - N_TRAIN_PER_SRC - N_VAL_PER_SRC
    print(f"Split config updated for {min_count} balanced rows / source:")
    print(f"  N_TRAIN_PER_SRC = {N_TRAIN_PER_SRC}  (~75 %)")
    print(f"  N_VAL_PER_SRC   = {N_VAL_PER_SRC}  (~12 %)")
    print(f"  N_TEST_PER_SRC  = {N_TEST_PER_SRC}  (~13 %)")
    print(f"  → {N_TRAIN_PER_SRC * len(datasets_raw)} total training examples")

print("\nSource sizes after capping:")
for src, rows in datasets_raw.items():
    print(f"  {src:<35} {len(rows):>5} rows")


Source sizes before capping:
  cfa_cpa                               768 rows
  es_multifin                           220 rows
  plutus                                246 rows
  arabic_accounting                     405 rows
  arabic_business                       374 rows
  hindi_finance                        5982 rows

Smallest source: 220 rows — capping all sources to this.
Split config updated for 220 balanced rows / source:
  N_TRAIN_PER_SRC = 165  (~75 %)
  N_VAL_PER_SRC   = 26  (~12 %)
  N_TEST_PER_SRC  = 29  (~13 %)
  → 990 total training examples

Source sizes after capping:
  cfa_cpa                               220 rows
  es_multifin                           220 rows
  plutus                                220 rows
  arabic_accounting                     220 rows
  arabic_business                       220 rows
  hindi_finance                         220 rows


In [7]:
# ── Cell 7 ──────────────────────────────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════
# Cell 5 — Hindi Translation (hindi_finance only)
# Translates question + all choices from Hindi → English.
# Why: Fin-R1/Qwen2.5 has minimal Hindi training data.
#      Hindi scored 34.7% in baseline — worst of all 6 sources.
#      Translating to English lets the model apply its strong English
#      financial reasoning to these questions.
# Arabic is NOT translated — model handles Arabic adequately (49–67%).
# ═══════════════════════════════════════════════════════════════════════════

if TRANSLATE_HINDI:
    from deep_translator import GoogleTranslator
    import time

    _translator = GoogleTranslator(source="hi", target="en")

    def _translate_text(text: str) -> str:
        """Translate one string Hindi→English with retry on rate limit."""
        if not text or not text.strip():
            return text
        for attempt in range(3):
            try:
                result = _translator.translate(text)
                return result if result else text
            except Exception as e:
                if attempt < 2:
                    time.sleep(2 ** attempt)
                else:
                    print(f"  [translate] failed after 3 attempts: {e}")
                    return text   # return original on failure

    def translate_row(row: dict) -> dict:
        """Translate question and all choices in a hindi_finance row."""
        translated = dict(row)
        translated["question"] = _translate_text(row["question"])
        translated["choices"]  = [_translate_text(c) for c in row["choices"]]
        translated["translated"] = True   # flag for debugging
        return translated

    print("Translating hindi_finance questions to English ...")
    hindi_rows = datasets_raw.get("hindi_finance", [])
    n = len(hindi_rows)

    translated_rows = []
    for i, row in enumerate(hindi_rows):
        translated_rows.append(translate_row(row))
        if (i + 1) % 50 == 0 or (i + 1) == n:
            print(f"  Translated {i+1}/{n} ...", flush=True)
        time.sleep(0.1)   # gentle rate limiting — Google Translate free tier

    datasets_raw["hindi_finance"] = translated_rows

    # Show a sample translation
    if translated_rows:
        orig  = hindi_rows[0]
        trans = translated_rows[0]
        print()
        print("Sample translation:")
        print(f"  Original : {orig['question'][:100]}")
        print(f"  English  : {trans['question'][:100]}")
    print(f"\nHindi translation complete: {n} questions translated.")

else:
    print("TRANSLATE_HINDI = False — skipping translation.")
    print("Hindi questions will be trained on in original Hindi script.")

Translating hindi_finance questions to English ...
  Translated 50/220 ...
  Translated 100/220 ...
  Translated 150/220 ...
  Translated 200/220 ...
  Translated 220/220 ...

Sample translation:
  Original : L, C से कैसे संबंधित है?
  English  : How is L related to C?

Hindi translation complete: 220 questions translated.


In [8]:
# ── Cell 8 ──────────────────────────────────────────────────────────────────
import random as _random

_SFT_RNG = _random.Random(RANDOM_SEED + 1)   # separate RNG for shuffling

_SYSTEM_PROMPT = (
    "You are a financial expert. For each multiple-choice question, reason through "
    "the options carefully inside <think>...</think> tags, then state your final "
    "answer inside <answer>...</answer> tags using only the answer letter."
)

# Arabic system prompt — keeps model in Arabic mode for Arabic questions
_SYSTEM_PROMPT_AR = (
    "أنت خبير مالي. لكل سؤال اختيار من متعدد، فكر في الخيارات بعناية داخل "
    "علامات <think>...</think>، ثم اذكر إجابتك النهائية داخل علامات "
    "<answer>...</answer> باستخدام حرف الإجابة فقط."
)


def format_choices(choices: list) -> tuple:
    labels = [chr(ord("A") + i) for i in range(len(choices))]
    opts   = "\n".join(f"{labels[i]}. {choices[i]}" for i in range(len(choices)))
    return opts, labels


def build_sft_example(row: dict, tokenizer, shuffle_choices: bool = True) -> dict | None:
    """
    Convert one MCQ row into an SFT training example.

    Key improvements over baseline:
    1. Choice shuffling: randomly reorders options so model can't memorise position.
    2. Language-aware CoT: uses Arabic system prompt for Arabic sources.
    3. Richer CoT template: references actual gold text, not just generic phrasing.
    """
    choices  = list(row["choices"])
    gold_raw = row["gold"]
    gold_key = gold_raw[0].lower()             # e.g. "b"
    gold_idx = row["keys"].index(gold_key)     # index in original order
    source   = row.get("source", "")

    # ── Choice shuffling ──────────────────────────────────────────────────
    if shuffle_choices and len(choices) > 1:
        order         = list(range(len(choices)))
        _SFT_RNG.shuffle(order)
        choices       = [choices[i] for i in order]
        new_gold_pos  = order.index(gold_idx)  # where gold ended up after shuffle
    else:
        new_gold_pos  = gold_idx

    gold_label = chr(ord("A") + new_gold_pos)   # e.g. "C" after shuffling
    gold_text  = choices[new_gold_pos]

    if new_gold_pos >= len(choices):
        return None   # safety check

    opts_str, labels = format_choices(choices)
    valid_str        = "/".join(labels)

    # ── System prompt — Arabic sources get Arabic system prompt ───────────
    is_arabic = source in {"arabic_accounting", "arabic_business"}
    sys_prompt = _SYSTEM_PROMPT_AR if is_arabic else _SYSTEM_PROMPT

    # ── User content ──────────────────────────────────────────────────────
    user_content = (
        f"Answer this multiple-choice financial question. "
        f"Valid answer letters: {valid_str}.\n\n"
        f"Question: {row['question']}\n\n{opts_str}"
    )

    # ── CoT reasoning — richer template referencing actual content ────────
    other_options = [
        f"{chr(ord('A') + i)}. {choices[i]}"
        for i in range(len(choices))
        if i != new_gold_pos
    ]
    others_str = "\n".join(other_options)

    think_content = (
        f"Let me analyze this financial question carefully.\n\n"
        f"Question: {row['question'][:200]}\n\n"
        f"Evaluating each option:\n"
        f"{gold_label}. {gold_text} — This is the correct answer. "
        f"It accurately reflects the financial principle being tested.\n\n"
        f"Eliminating the other options:\n{others_str}\n"
        f"These are incorrect because they either misstate the concept, "
        f"provide incomplete information, or contradict standard financial principles.\n\n"
        f"Therefore, the correct answer is {gold_label}."
    )

    messages = [
        {"role": "system",    "content": sys_prompt},
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": (
            f"<think>\n{think_content}\n</think>\n"
            f"<answer>\n{gold_label}\n</answer>"
        )},
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {
        "text":     text,
        "source":   source,
        "gold":     gold_label,
        "shuffled": shuffle_choices,
    }


print("SFT formatter defined.")
print(f"  Choice shuffling : {SHUFFLE_CHOICES}")
print(f"  Arabic prompt    : Yes (arabic_accounting, arabic_business)")
print(f"  Hindi source     : {'translated to English' if TRANSLATE_HINDI else 'original Hindi'}")

SFT formatter defined.
  Choice shuffling : True
  Arabic prompt    : Yes (arabic_accounting, arabic_business)
  Hindi source     : translated to English


In [9]:
# ── Cell 9 ──────────────────────────────────────────────────────────────────

import random
from collections import defaultdict

rng = random.Random(RANDOM_SEED)

train_rows, val_rows, test_rows = [], [], []

# ── Guard: fail fast if no data was loaded ────────────────────────────────
if not datasets_raw:
    raise RuntimeError(
        "datasets_raw is empty — all dataset sources failed to load.\n"
        "Check that GITHUB_TOKEN env var is set and the repo/files are accessible.\n"
        "Look for errors printed by the dataset loading cell above."
    )

print("Splitting datasets:")
print(f"{'Source':<33}  {'Train':>6}  {'Val':>6}  {'Test':>6}  {'Total':>6}")
print("-" * 65)

for src_name, rows in datasets_raw.items():
    shuffled = rows[:]
    rng.shuffle(shuffled)

    n_train = min(N_TRAIN_PER_SRC, len(shuffled))
    n_val   = min(N_VAL_PER_SRC,   len(shuffled) - n_train)
    n_test  = min(N_TEST_PER_SRC,  len(shuffled) - n_train - n_val)

    train_rows.extend(shuffled[:n_train])
    val_rows.extend(shuffled[n_train: n_train + n_val])
    test_rows.extend(shuffled[n_train + n_val: n_train + n_val + n_test])

    print(f"  {src_name:<31}  {n_train:>6}  {n_val:>6}  {n_test:>6}  {len(rows):>6}")

print("-" * 65)
print(f"  {'TOTAL':<31}  {len(train_rows):>6}  {len(val_rows):>6}  {len(test_rows):>6}")

# ── Guard: fail fast if split produced no training examples ───────────────
if len(train_rows) == 0:
    raise RuntimeError(
        f"train_rows is empty after splitting {len(datasets_raw)} source(s).\n"
        "Each source may have had fewer rows than N_TRAIN_PER_SRC. "
        "Check the totals printed above."
    )

# Shuffle splits globally
rng.shuffle(train_rows)
rng.shuffle(val_rows)
rng.shuffle(test_rows)

print()
print(f"Training examples   : {len(train_rows)}")
print(f"Validation examples : {len(val_rows)}")
print(f"Test examples       : {len(test_rows)}  (held out — only used in Cell 10)")
if TEST_MODE:
    print()
    print("  ⚠  TEST_MODE: very small splits — expected for smoke-test")


Splitting datasets:
Source                              Train     Val    Test   Total
-----------------------------------------------------------------
  cfa_cpa                             165      26      29     220
  es_multifin                         165      26      29     220
  plutus                              165      26      29     220
  arabic_accounting                   165      26      29     220
  arabic_business                     165      26      29     220
  hindi_finance                       165      26      29     220
-----------------------------------------------------------------
  TOTAL                               990     156     174

Training examples   : 990
Validation examples : 156
Test examples       : 174  (held out — only used in Cell 10)


In [10]:
# ── Cell 10 ──────────────────────────────────────────────────────────────────

# ── Verify choice distribution in training set ─────────────────────────────────────────
# This cell checks that gold answer positions are roughly uniform after shuffling.
# If SHUFFLE_CHOICES=False, you'll see the original bias (e.g., 38% for position A).
# If SHUFFLE_CHOICES=True, all positions should be within 5% of each other.

from collections import Counter
import random as _random

def check_gold_distribution(rows, label="train"):
    """Check distribution of gold answer positions."""
    if not rows:
        print(f"\nGold answer distribution in {label} set (0 examples):")
        print(f"  ⚠  No rows to check — dataset loading may have failed.")
        print(f"     Check that GITHUB_TOKEN is set and the dataset files are accessible.")
        raise RuntimeError(
            f"'{label}' split has 0 examples. "
            "Run the dataset loading cell again and check for errors."
        )

    # Build quick SFT examples without tokenizer to check positions
    gold_positions = []
    for row in rows:
        choices  = list(row["choices"])
        gold_key = row["gold"][0].lower()
        gold_idx = row["keys"].index(gold_key)
        if SHUFFLE_CHOICES:
            order = list(range(len(choices)))
            _SFT_RNG_CHECK = _random.Random(RANDOM_SEED + 999)  # separate seed for check
            _SFT_RNG_CHECK.shuffle(order)
            new_pos = order.index(gold_idx)
        else:
            new_pos = gold_idx
        gold_positions.append(chr(ord("A") + new_pos))

    counts = Counter(gold_positions)
    total  = len(gold_positions)
    print(f"\nGold answer distribution in {label} set ({total} examples):")
    for letter in sorted(counts):
        pct  = counts[letter] / total * 100
        bar  = "█" * int(pct / 2)
        print(f"  {letter}: {counts[letter]:>5} ({pct:5.1f}%)  {bar}")
    max_pct = max(counts[l] / total * 100 for l in counts)
    if max_pct > 40:
        print(f"  ⚠  WARNING: positional bias detected ({max_pct:.1f}% for one letter)")
    else:
        print(f"  ✓ Distribution looks balanced (max {max_pct:.1f}%)")

check_gold_distribution(train_rows, "train")



Gold answer distribution in train set (990 examples):
  A:   233 ( 23.5%)  ███████████
  B:   284 ( 28.7%)  ██████████████
  C:   207 ( 20.9%)  ██████████
  D:    95 (  9.6%)  ████
  E:   119 ( 12.0%)  ██████
  F:    52 (  5.3%)  ██
  ✓ Distribution looks balanced (max 28.7%)


In [11]:
# ── Cell 11 ──────────────────────────────────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from datasets import Dataset

# ── No quantization — load straight in BF16 ───────────────────────────────
# A100 40GB has plenty of VRAM for 7B in BF16 (~14 GB) + LoRA adapters.
# This avoids ALL bitsandbytes CUDA alignment/cuBLAS errors entirely.

print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Loading model in BF16 (no quantization) ...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,   # T4: fp16 only — for A100 use torch.bfloat16
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",  # disable flash-attn — not needed on T4
)
model.config.use_cache = False
model.enable_input_require_grads()  # required for gradient checkpointing with PEFT

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

allocated = torch.cuda.memory_allocated() / 1e9
print(f"VRAM used after model load: {allocated:.1f} GB")

# ── Build HuggingFace datasets ─────────────────────────────────────────────
print("\nBuilding SFT examples ...")

def make_hf_dataset(rows, tok):
    examples = []
    skipped  = 0
    for row in rows:
        ex = build_sft_example(row, tok, shuffle_choices=SHUFFLE_CHOICES)
        if ex is None:
            skipped += 1
            continue
        examples.append(ex)
    if skipped:
        print(f"  Skipped {skipped} malformed rows")
    return Dataset.from_list(examples)

train_dataset = make_hf_dataset(train_rows, tokenizer)
val_dataset   = make_hf_dataset(val_rows,   tokenizer)

print(f"Train dataset : {len(train_dataset)} examples")
print(f"Val dataset   : {len(val_dataset)} examples")
print()
print("=== SAMPLE TRAINING EXAMPLE ===")
print(train_dataset[0]["text"][:800])
print("...")


Loading tokenizer: SUFE-AIFLM-Lab/Fin-R1
Loading model in BF16 (no quantization) ...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 161,480,704 || all params: 7,777,097,216 || trainable%: 2.0764
VRAM used after model load: 16.4 GB

Building SFT examples ...
Train dataset : 990 examples
Val dataset   : 156 examples

=== SAMPLE TRAINING EXAMPLE ===
<|im_start|>system
You are a financial expert. For each multiple-choice question, reason through the options carefully inside <think>...</think> tags, then state your final answer inside <answer>...</answer> tags using only the answer letter.<|im_end|>
<|im_start|>user
Answer this multiple-choice financial question. Valid answer letters: A/B/C/D/E/F.

Question: Διάβασε το κείμενο προσεκτικά και επέλεξε την σωστή κατηγοριά για το κείμενο από τις κατηγορίες Φορολογία & Λογιστική, Επιχειρήσεις & Διοίκηση, Οικονομικά, Βιομηχανία, Τεχνολογία, Κυβέρνηση & Έλεγχοι. Κείμενο: Tα σημαντικότερα ΔΠΧΑ (IFRS) και ο χειρισμός στη Φορολογία Εισοδήματος και στη Διανομή Κερδών. Απάντηση:

A. Κυβέρνηση & Έλεγχοι
B. Τεχνολογία
C. Βιομηχανία
D. Επιχειρήσεις & Διοίκηση
E. Οικο

In [12]:
# ── Cell 12 ──────────────────────────────────────────────────────────────────
import inspect
from trl import SFTTrainer, SFTConfig

# ── Figure out where each param lives in this TRL version ──────────────────
# Older TRL: dataset_text_field / max_seq_length / packing live on SFTTrainer
# Newer TRL: they moved into SFTConfig
_cfg_params     = inspect.signature(SFTConfig.__init__).parameters
_trainer_params = inspect.signature(SFTTrainer.__init__).parameters

def _route(key, value, cfg_kw, trainer_kw):
    if key in _cfg_params:
        cfg_kw[key] = value
    elif key in _trainer_params:
        trainer_kw[key] = value
    # silently drop if neither accepts it (shouldn't happen)

cfg_kw     = {}
trainer_kw = {}

for k, v in [
    ("dataset_text_field", "text"),
    ("max_seq_length",     MAX_SEQ_LEN),
    ("packing",            False),
]:
    _route(k, v, cfg_kw, trainer_kw)

# ── Training arguments — clean and simple ──────────────────────────────────
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO,
    bf16=BF16,
    fp16=FP16,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=False,
    report_to="none",
    remove_unused_columns=True,
    gradient_checkpointing=True,   # saves ~30% VRAM — essential for 7B BF16
    optim="adamw_torch",              # avoids fused kernels that cause misaligned address with bnb 4-bit
    **cfg_kw,
)

# ── Tokenizer kwarg name also changed across versions ──────────────────────
_tok_key = "processing_class" if "processing_class" in _trainer_params else "tokenizer"

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    **{_tok_key: tokenizer},
    **trainer_kw,
)

eff_batch       = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = max(1, len(train_dataset) // eff_batch)
total_steps     = steps_per_epoch * EPOCHS if MAX_STEPS == -1 else MAX_STEPS

print("Starting training ...")
print(f"  Steps per epoch : {steps_per_epoch}")
print(f"  Total steps     : {total_steps}")
if TEST_MODE:
    print("  ⚠  TEST_MODE: running only 4 steps")
print()

train_result = trainer.train()

allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved() / 1e9
print(f"  GPU memory allocated : {allocated:.1f} GB")
print(f"  GPU memory reserved  : {reserved:.1f} GB")
print()
print("Training complete.")
print(f"  Train loss   : {train_result.training_loss:.4f}")
print(f"  Runtime      : {train_result.metrics.get('train_runtime', 0):.0f}s")


Map:   0%|          | 0/990 [00:00<?, ? examples/s]

Map:   0%|          | 0/156 [00:00<?, ? examples/s]

Starting training ...
  Steps per epoch : 30
  Total steps     : 90



/opt/conda/lib/python3.10/site-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss,Validation Loss
50,0.330400,0.341753


/opt/conda/lib/python3.10/site-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


  GPU memory allocated : 17.7 GB
  GPU memory reserved  : 24.5 GB

Training complete.
  Train loss   : 0.4492
  Runtime      : 839s


In [13]:
# ── Cell 13 ──────────────────────────────────────────────────────────────────
import gc
import json as _json

# ── Save LoRA adapter to Google Drive (always — survives session disconnect) ──
adapter_path = os.path.join(OUTPUT_DIR, "lora_adapter")
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"LoRA adapter saved: {adapter_path}")
print("(Load later: PeftModel.from_pretrained(base_model, adapter_path))")

# ── Save training config for reproducibility ──────────────────────────────────
config_dict = {
    "model_id":         MODEL_ID,
    "test_mode":        TEST_MODE,
    "lora_r":           LORA_R,
    "lora_alpha":       LORA_ALPHA,
    "lora_dropout":     LORA_DROPOUT,
    "target_modules":   LORA_TARGET_MODULES,
    "epochs":           EPOCHS,
    "max_steps":        MAX_STEPS,
    "batch_size":       BATCH_SIZE,
    "grad_accum":       GRAD_ACCUM,
    "effective_batch":  BATCH_SIZE * GRAD_ACCUM,
    "learning_rate":    LEARNING_RATE,
    "lr_scheduler":     LR_SCHEDULER,
    "max_seq_len":      MAX_SEQ_LEN,
    "n_train_per_src":  N_TRAIN_PER_SRC,
    "n_val_per_src":    N_VAL_PER_SRC,
    "n_test_per_src":   N_TEST_PER_SRC,
    "train_total":      len(train_dataset),
    "val_total":        len(val_dataset),
    "train_loss":       round(train_result.training_loss, 4),
    "random_seed":      RANDOM_SEED,
}
config_path = os.path.join(OUTPUT_DIR, "finetune_config.json")
with open(config_path, "w") as f:
    _json.dump(config_dict, f, indent=2)
print(f"Config saved: {config_path}")

print("\nAdapter saved. Run Cell 11 (eval) before pushing to Hub.")

LoRA adapter saved: /workspace/finr1_finetuned/lora_adapter
(Load later: PeftModel.from_pretrained(base_model, adapter_path))
Config saved: /workspace/finr1_finetuned/finetune_config.json

Adapter saved. Run Cell 11 (eval) before pushing to Hub.


In [14]:
# ── Cell 14 ──────────────────────────────────────────────────────────────────
# Evaluation — reload adapter, run on held-out test set
import re, gc
import torch
from peft import PeftModel
from collections import defaultdict
from transformers import AutoModelForCausalLM

for _var in ["trainer"]:
    if _var in globals(): del globals()[_var]
gc.collect()
torch.cuda.empty_cache()

print("Loading fine-tuned model for evaluation ...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True,
)
ft_model = PeftModel.from_pretrained(base_model, adapter_path)
ft_model.eval()
tokenizer.padding_side = "left"
print("Fine-tuned model loaded.")

def _extract_answer(text, labels):
    import re as _re
    m = _re.search(r"<answer>\s*([A-Za-z])\s*</answer>", text)
    if m and m.group(1).upper() in labels: return m.group(1).upper()
    m = _re.search(r"Answer\s*:\s*([A-Za-z])", text, _re.IGNORECASE)
    if m and m.group(1).upper() in labels: return m.group(1).upper()
    for ch in text.upper():
        if ch in labels: return ch
    return None

def evaluate_on_rows(model, tokenizer, rows, max_new_tokens=256, eval_batch_size=4):
    correct_by_src, total_by_src = defaultdict(int), defaultdict(int)
    for batch_start in range(0, len(rows), eval_batch_size):
        batch = rows[batch_start: batch_start + eval_batch_size]
        prompts = []
        for row in batch:
            ex = build_sft_example(row, tokenizer, shuffle_choices=False)
            prompts.append(ex["text"].split("<|im_start|>assistant")[0] + "<|im_start|>assistant\n")
        enc = tokenizer(prompts, return_tensors="pt", padding=True,
                        truncation=True, max_length=MAX_SEQ_LEN).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=tokenizer.pad_token_id)
        input_len = enc["input_ids"].shape[1]
        for b_idx, row in enumerate(batch):
            text   = tokenizer.decode(out[b_idx, input_len:], skip_special_tokens=True).strip()
            pred   = _extract_answer(text, ["A","B","C","D","E"])
            gold   = row.get("gold") or row.get("answer") or ["?"]
            gold   = [g.upper() for g in (gold if isinstance(gold, list) else [gold])]
            total_by_src[row["source"]]   += 1
            correct_by_src[row["source"]] += int(pred in gold if pred else False)
        del enc, out
        torch.cuda.empty_cache()
        done_c = sum(correct_by_src.values())
        done_t = sum(total_by_src.values())
        print(f"  [{done_t:>4}/{len(rows)}]  running acc: {done_c/done_t*100:.1f}%", flush=True)
    res = {src: {"correct": correct_by_src[src], "total": total_by_src[src],
                 "acc": round(correct_by_src[src]/total_by_src[src]*100,1)}
           for src in total_by_src}
    tc, tt = sum(correct_by_src.values()), sum(total_by_src.values())
    res["OVERALL"] = {"correct": tc, "total": tt, "acc": round(tc/tt*100,1) if tt else 0.0}
    return res

print(f"Evaluating on {len(test_rows)} held-out test examples ...")
if TEST_MODE: print("  ☢  TEST_MODE: very few examples — accuracy not meaningful")
print()
results = evaluate_on_rows(ft_model, tokenizer, test_rows)

print()
print(f"{'Source':<33}  {'Correct':>8}  {'Total':>6}  {'Accuracy':>9}")
print("-"*65)
for src, r in results.items():
    marker = "  ┈ OVERALL" if src == "OVERALL" else ""
    print(f"  {src:<31}  {r['correct']:>8}  {r['total']:>6}  {r['acc']:>8.1f}%{marker}")

import json as _j
eval_path = os.path.join(OUTPUT_DIR, "eval_results.json")
with open(eval_path, "w") as f: _j.dump(results, f, indent=2)
print(f"\nEval results saved: {eval_path}")


Loading fine-tuned model for evaluation ...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Fine-tuned model loaded.
Evaluating on 174 held-out test examples ...



/opt/conda/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  [   4/174]  running acc: 50.0%
  [   8/174]  running acc: 62.5%
  [  12/174]  running acc: 50.0%
  [  16/174]  running acc: 43.8%
  [  20/174]  running acc: 45.0%
  [  24/174]  running acc: 50.0%
  [  28/174]  running acc: 42.9%
  [  32/174]  running acc: 40.6%
  [  36/174]  running acc: 44.4%
  [  40/174]  running acc: 47.5%
  [  44/174]  running acc: 45.5%
  [  48/174]  running acc: 41.7%
  [  52/174]  running acc: 42.3%
  [  56/174]  running acc: 41.1%
  [  60/174]  running acc: 38.3%
  [  64/174]  running acc: 37.5%
  [  68/174]  running acc: 36.8%
  [  72/174]  running acc: 37.5%
  [  76/174]  running acc: 36.8%
  [  80/174]  running acc: 36.2%
  [  84/174]  running acc: 34.5%
  [  88/174]  running acc: 35.2%
  [  92/174]  running acc: 35.9%
  [  96/174]  running acc: 35.4%
  [ 100/174]  running acc: 36.0%
  [ 104/174]  running acc: 35.6%
  [ 108/174]  running acc: 36.1%
  [ 112/174]  running acc: 36.6%
  [ 116/174]  running acc: 36.2%
  [ 120/174]  running acc: 36.7%
  [ 124/17

In [ ]:
# ── Cell 15 ──────────────────────────────────────────────────────────────────
# Push to Hub — ALWAYS deletes first so repo is always clean and loadable
import gc, re
import torch
from peft import PeftModel
from huggingface_hub import delete_repo, ModelCard
from transformers import AutoModelForCausalLM, AutoTokenizer

if not HUB_REPO:
    print("HUB_REPO is empty — skipping. Set it in Cell 2 and re-run.")
else:
    # ── 1. Delete existing repo (clean slate every time) ─────────────────
    print(f"Step 1 — Deleting existing Hub repo: {HUB_REPO} ...")
    try:
        delete_repo(repo_id=HUB_REPO, repo_type="model")
        print("  Deleted.")
    except Exception as e:
        print(f"  Nothing to delete ({e})")

    # ── 2. Free GPU ───────────────────────────────────────────────────────
    for _var in ["ft_model", "base_model", "trainer"]:
        if _var in globals(): del globals()[_var]
    gc.collect()
    torch.cuda.empty_cache()
    print("  GPU freed.")

    # ── 3. Reload base + adapter, merge to fp16, push ────────────────────
    print(f"\nStep 2 — Merging adapter and pushing to Hub ...")
    base_for_merge = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16,
        device_map="cpu", trust_remote_code=True,
    )
    merged   = PeftModel.from_pretrained(base_for_merge, adapter_path).merge_and_unload()
    push_tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    merged.push_to_hub(HUB_REPO, safe_serialization=True,
                       commit_message="Clean push: QLoRA merged fp16")
    push_tok.push_to_hub(HUB_REPO, commit_message="Add tokenizer")
    print(f"  Pushed: https://huggingface.co/{HUB_REPO}")

    # ── 4. Push model card ────────────────────────────────────────────────
    eval_rows = "\n".join(
        f"| {src} | {r['acc']}% | {r['correct']}/{r['total']} |"
        for src, r in results.items()
    )
    card_text = (
        "---\n"
        "license: apache-2.0\n"
        f"base_model: {MODEL_ID}\n"
        "tags:\n- finance\n- multiple-choice\n- fine-tuned\n- qwen2\n- lora\n"
        "---\n\n"
        f"# Fin-R1 Fine-Tuned — Multilingual Financial MCQ\n\n"
        "Merged full model (base + LoRA). Load directly — no adapter needed.\n\n"
        "## Usage\n"
        "```python\n"
        "from transformers import AutoModelForCausalLM, AutoTokenizer\n"
        f'model = AutoModelForCausalLM.from_pretrained("{HUB_REPO}", torch_dtype="auto")\n'
        f'tokenizer = AutoTokenizer.from_pretrained("{HUB_REPO}")\n'
        "```\n\n"
        "## Evaluation Results\n"
        "| Source | Accuracy | Correct/Total |\n"
        "|---|---|---|\n"
        f"{eval_rows}\n"
    )
    ModelCard(card_text).push_to_hub(HUB_REPO)
    print("  Model card pushed.")

    # ── 5. Free merged model ──────────────────────────────────────────────
    del merged, base_for_merge, push_tok
    gc.collect()
    torch.cuda.empty_cache()

    # ── 6. Reload from Hub exactly as any user would ──────────────────────
    print(f"\nStep 3 — Reloading from Hub (simulating a new user) ...")
    hub_model = AutoModelForCausalLM.from_pretrained(
        HUB_REPO, torch_dtype=torch.float16,
        device_map="auto", trust_remote_code=True,
    )
    hub_tok = AutoTokenizer.from_pretrained(HUB_REPO, trust_remote_code=True)
    hub_model.eval()
    hub_tok.padding_side = "left"
    print("  Hub model loaded successfully.")

    # ── 7. Quick prediction ───────────────────────────────────────────────
    print("\nStep 4 — Running a prediction to verify ...")
    sample  = test_rows[0]
    ex      = build_sft_example(sample, hub_tok, shuffle_choices=False)
    prompt  = ex["text"].split("<|im_start|>assistant")[0] + "<|im_start|>assistant\n"
    inputs  = hub_tok(prompt, return_tensors="pt",
                      truncation=True, max_length=MAX_SEQ_LEN).to(hub_model.device)
    with torch.no_grad():
        out = hub_model.generate(**inputs, max_new_tokens=150, do_sample=False,
                                 pad_token_id=hub_tok.pad_token_id)
    generated = hub_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    gold = sample.get("gold") or sample.get("answer") or ["?"]
    if isinstance(gold, str): gold = [gold]
    m    = re.search(r"<answer>\s*([A-E])", generated, re.IGNORECASE)
    pred = m.group(1).upper() if m else (generated.strip()[0].upper() if generated.strip() else "?")

    print(f"  Gold: {gold}  |  Predicted: {pred}  |  {'✓ CORRECT' if pred in gold else '✗ WRONG'}")
    print(f"  Output: {generated[:150]}")
    print()
    print("✅ Hub repo is clean and verified.")
    print(f"   Load with:")
    print(f'   AutoModelForCausalLM.from_pretrained("{HUB_REPO}", torch_dtype=torch.float16)')


Step 1 — Deleting existing Hub repo: hassanshahzad2003/finr1-mcq-multilingual ...
  Deleted.
  GPU freed.

Step 2 — Merging adapter and pushing to Hub ...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

Upload 4 LFS files:   0%|          | 0/4 [00:00<?, ?it/s]

## ✅ Pipeline Complete

| Cell | What it does |
|------|-------------|
| 0 | Install pinned deps + GPU cleanup — run once, then restart kernel |
| 2 | Config — edit `MODEL_ID`, `HUB_REPO`, `TEST_MODE` here |
| 3 | Secondary installs |
| 4 | HuggingFace + Google Drive login |
| 5 | Load datasets from GitHub |
| 6 | Cap rows per source (TEST_MODE) |
| 7 | Translate Hindi → English |
| 8 | Define SFT formatter |
| 9 | Train/val/test split |
| 10 | Check label distribution |
| 11 | Load model + build datasets |
| 12 | Train |
| 13 | Save LoRA adapter to Drive |
| 14 | Evaluate on held-out test set |
| 15 | **Delete → merge → push → reload → verify** |


## Next Steps

### Load the fine-tuned adapter in the Kaggle inference notebook

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.float16)
base_model = AutoModelForCausalLM.from_pretrained(
    "SUFE-AIFLM-Lab/Fin-R1", quantization_config=bnb_config, device_map="auto"
)
model = PeftModel.from_pretrained(base_model, "/path/to/lora_adapter")
model.eval()
```

### Merge adapter into base model (optional — faster inference, no PEFT dependency)

```python
merged = model.merge_and_unload()
merged.save_pretrained("finr1_mcq_merged")
```

### Troubleshooting

| Symptom | Fix |
|---|---|
| OOM on T4 | Reduce `BATCH_SIZE` to 1, `MAX_SEQ_LEN` to 768 |
| Loss not decreasing | Lower `LEARNING_RATE` to `1e-4`, check sample format in Cell 7 |
| Adapter not loading | Ensure `adapter_config.json` exists in `lora_adapter/` folder |
| Eval accuracy lower than baseline | Reduce epochs to 1 or reduce `N_TRAIN_PER_SRC` to 100 |

### Recommended training data amounts

| Scenario | `N_TRAIN_PER_SRC` | Total |
|---|---|---|
| Verify pipeline (TEST_MODE) | 2 | 12 |
| Quick sanity check | 50 | ~300 |
| Standard adaptation | 200 | ~1200 |
| Maximum useful | 300 | ~1800 |
| Overfitting risk above | 400+ | ~2400+ |

## Running on Vast.ai

### Before starting the instance, set secrets:
```bash
# In your local terminal, before SSHing in:
export HF_TOKEN="your_huggingface_token"
export GITHUB_TOKEN="your_github_token"
```

### Start the instance and run:
```bash
# SSH into instance
ssh -p PORT root@HOST

# Set env vars
export HF_TOKEN="your_hf_token"
export GITHUB_TOKEN="your_github_token"

# Install jupyter and run notebook
pip install jupyter
jupyter nbconvert --to notebook --execute \
  --ExecutePreprocessor.timeout=7200 \
  finr1_finetune_vastai.ipynb \
  --output finr1_finetune_vastai_output.ipynb
```

### After training completes:
```bash
# Download the output notebook to inspect results
scp -P PORT root@HOST:/workspace/finr1_finetune_vastai_output.ipynb ./
```

### Cost estimate:
- A100 SXM4 40GB at $0.60/hr
- Full training run: ~30 minutes = ~$0.30 total